# Transportation & Logistics Tracking — Enterprise Analytics**Purpose:** End-to-end analytics for logistics management — route performance, supplier efficiency, material lead times, delay root-cause analysis, and ML-based ETA/delay/risk models.**Dataset:** `Transportation & Logistics Tracking Dataset.xlsx` (Primary Data + Data Dictionary)**Audience:** Executives, fleet managers, operations, supply chain analysts.

In [ ]:
import sysfrom pathlib import Pathimport warningswarnings.filterwarnings('ignore')ROOT = Path.cwd()if not (ROOT / 'src').exists():    ROOT = ROOT.parentsys.path.insert(0, str(ROOT))import pandas as pdimport numpy as npimport plotly.io as piofrom IPython.display import display, Markdown, HTMLfrom src.data_loader import load_shipments, load_raw, profile_datasetfrom src.cleaning import clean_shipmentsfrom src.features import engineer_featuresfrom src.gps import snapshot_gps_summary, preprocess_gps_trajectoryfrom src import analytics, root_cause, ml_models, vizpio.renderers.default = 'notebook'print('Project root:', ROOT)

## 1. Automated Dataset Intelligence

In [ ]:
raw, data_dict = load_raw()display(data_dict.head(15))ship_raw = load_shipments()profile = profile_dataset(ship_raw)display(Markdown('### Column meanings & dtypes'))display(pd.DataFrame({'column': list(profile['dtypes'].keys()), 'dtype': list(profile['dtypes'].values())}))display(Markdown('### Missing values (%)'))display(pd.Series(profile['missing_pct']).sort_values(ascending=False))display(Markdown('### Shipment lifecycle & GPS behavior'))display(pd.DataFrame([{    'gps_mode': profile['gps_tracking_mode'],    'median_pings_per_booking': profile['gps_pings_per_booking']['50%'],    'booking_date_range': f"{profile['date_range']['booking_min']} → {profile['date_range']['booking_max']}",}]).T)display(Markdown('### Operational workflow (inferred)'))display(Markdown('''1. **Booking** (`booking_date`) — contract/spot shipment created  2. **Dispatch wait** — gap until `trip_start` (loading/planning)  3. **In-transit** — `trip_start` → `trip_end` (moving time)  4. **GPS snapshot** — `gps_ping_time` + current coordinates (last known)  5. **Planned vs actual arrival** — `planned_eta` vs `actual_eta` → `ontime_flag`  6. **Unloading** — residual after `trip_end` if delivery completes later  '''))display(Markdown('### Potential KPIs'))display(pd.DataFrame({'kpi': profile['potential_kpis']}))display(Markdown('### Data quality flags'))display(pd.DataFrame({'flag': profile['data_quality_flags']}))

## 2. Data Cleaning Pipeline

In [ ]:
df, qlog = clean_shipments(ship_raw)df = engineer_features(df)print('Clean rows:', len(df))display(qlog)display(df.head(3))

## 3. Executive KPIs (Power BI Overall Dashboard)

In [ ]:
kpis = analytics.executive_kpis(df)display(HTML(viz.kpi_cards_html(kpis)))display(kpis)

## 4. GPS Tracking Behavior & Preprocessing

In [ ]:
display(snapshot_gps_summary(df))# Trajectory preprocessor ready for high-frequency GPS feedssample_multi = ship_raw[ship_raw.duplicated('Booking ID', keep=False)]if len(sample_multi):    pings = sample_multi.rename(columns={        'Booking ID':'booking_id','Data Ping time':'gps_ping_time',        'Current Location Latitude':'current_lat','Curren Location Longitude':'current_lon'    })    display(preprocess_gps_trajectory(pings).head())else:    print('Note: Current file is snapshot GPS (~1 ping/booking). Enterprise deployment needs 30-60s pings for idle/moving decomposition.')

## 5. On-Time Delivery & Delay Analytics

In [ ]:
display(analytics.delay_patterns(df).tail(12))fig = viz.bookings_trend(df)fig.show()fig2 = viz.origin_bubble_map(df)fig2.show()

## 6. Route Performance & Inefficiency

In [ ]:
routes = analytics.route_kpis(df, min_bookings=10)display(routes.head(20).style.background_gradient(subset=['ontime_rate_pct'], cmap='RdYlGn'))# High delay + high volume routes = priority fixespriority = routes[(routes['avg_delay_hours']>48) & (routes['total_bookings']>=20)].head(15)display(Markdown('### Priority routes (high delay + volume)'))display(priority)

## 7. Supplier Performance — Segmented (Not Overall Average)

In [ ]:
sup_overall, sup_segment = analytics.supplier_performance_segmented(df)display(sup_overall.head(15))fig = viz.supplier_combo_chart(sup_overall)fig.show()# Example RCA for a large supplierexample_sup = sup_overall.iloc[0]['supplier_name']display(Markdown(f'### Capability gap vs peers: **{example_sup}**'))display(root_cause.supplier_capability_gap(df, example_sup))

## 8. Customer Shipment Visibility

In [ ]:
customers = analytics.segment_performance(df, 'customer_name', min_bookings=20)display(customers.head(15).style.background_gradient(subset=['ontime_rate_pct'], cmap='RdYlGn'))

## 9. Material / Goods Type vs Delivery Time

In [ ]:
mat = analytics.material_lead_time(df)display(mat.head(20))import plotly.express as pxpx.scatter(mat, x='median_trip_days', y='ontime_rate', size='bookings', hover_name='material_group',           title='Material: Lead Time vs On-time Rate').show()

## 10. Lead Time Decomposition (Wait / Moving / Unloading / Idle)

In [ ]:
display(root_cause.bottleneck_diagnostics(df))lead = df[['booking_id','wait_hours','moving_hours','unloading_hours','idle_days_proxy','delay_hours']].describe()display(lead)

## 11. Root-Cause Analysis (Multivariate — Not Correlation Only)

In [ ]:
delay_model = root_cause.delay_driver_model(df)display(Markdown(f"Delay model MAE: **{delay_model['test_mae_hours']} hours**"))display(delay_model['top_drivers'])late_model = root_cause.late_risk_classifier(df)display(Markdown(f"Late-risk classifier AUC: **{late_model['test_auc']}**"))display(late_model['top_risk_drivers'])display(Markdown(delay_model['interpretation_note']))

## 12. Fleet, Driver & Vehicle Segmentation

In [ ]:
display(analytics.fleet_utilization_proxy(df).head(12))display(analytics.driver_scorecard(df).head(12))# Vehicle type × distance bandseg_vt = analytics.segment_performance(df, ['vehicle_type','distance_band'], min_bookings=5)display(seg_vt.head(20))

## 13. Machine Learning Models

In [ ]:
eta = ml_models.train_eta_model(df)delay_clf = ml_models.train_delay_classifier(df)df['risk_score'] = ml_models.shipment_risk_score(df, delay_clf)anomalies = ml_models.route_anomaly_detection(df)forecast = ml_models.forecast_monthly_volume(df)display(Markdown(f"**ETA (trip days) MAE:** {eta['test_mae_days']} "))display(Markdown(f"**Delay classifier AUC:** {delay_clf['test_auc']}"))display(df[['booking_id','supplier_name','route_id','risk_score','is_late']].sort_values('risk_score', ascending=False).head(15))display(Markdown('### Route anomalies'))display(anomalies[anomalies['is_anomaly']].head(10))display(Markdown('### Volume forecast (next 3 months)'))display(forecast)

## 14. Management Recommendations — Logistics Transformation Roadmap

In [ ]:
from src.recommendations import generate_management_recommendations, recommendations_markdown_tablemgmt_recs = generate_management_recommendations(df, delay_model=delay_model)display(mgmt_recs)  # structured initiative registerdisplay(Markdown(recommendations_markdown_table(mgmt_recs)))